# ATLAS R3 — MMLU + LegalBench Benchmark
**Model:** `Rafaelcedav/atlas-r3-qwen3-14b`  
**Runtime:** Google Colab T4 (free) + GGUF Q4_K_M quantization  
**Benchmarks:** MMLU (professional_law, professional_accounting, business_ethics) + LegalBench subsets

> Run cells top to bottom. Total time ~60-90 min on T4.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install -q lm_eval datasets huggingface_hub tqdm

In [ ]:
# ── Cell 2: Download GGUF model from HuggingFace ───────────────────────────────
# We use a Q4_K_M quantized version. If atlas-r3 has no GGUF yet,
# we quantize on the fly from the safetensors (requires ~20 GB disk).

from huggingface_hub import snapshot_download, hf_hub_download
import os

HF_TOKEN   = "hf_REPLACE_WITH_YOUR_TOKEN"  # <-- paste your HF token here
MODEL_ID   = "Rafaelcedav/atlas-r3-qwen3-14b"
GGUF_DIR   = "/content/atlas_r3_gguf"

os.makedirs(GGUF_DIR, exist_ok=True)

# Try to download a pre-quantized GGUF if available
try:
    gguf_path = hf_hub_download(
        repo_id=MODEL_ID,
        filename="atlas-r3-q4_k_m.gguf",
        token=HF_TOKEN,
        local_dir=GGUF_DIR,
    )
    print(f"Downloaded GGUF: {gguf_path}")
except Exception:
    print("GGUF not found in repo — will quantize from safetensors.")
    gguf_path = None

In [ ]:
# ── Cell 3: Quantize if GGUF not pre-built ─────────────────────────────────────
# Skip this cell if Cell 2 downloaded the GGUF successfully.

if gguf_path is None:
    from huggingface_hub import snapshot_download

    print("Downloading safetensors model (~28 GB)...")
    model_dir = snapshot_download(repo_id=MODEL_ID, token=HF_TOKEN, local_dir="/content/atlas_r3_hf")

    print("Installing llama.cpp converter...")
    !git clone -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !pip install -q -r /content/llama.cpp/requirements.txt

    print("Converting to GGUF F16...")
    !python /content/llama.cpp/convert_hf_to_gguf.py /content/atlas_r3_hf \
        --outfile {GGUF_DIR}/atlas-r3-f16.gguf --outtype f16

    print("Quantizing to Q4_K_M (~9 GB)...")
    !{GGUF_DIR}/../llama.cpp/llama-quantize \
        {GGUF_DIR}/atlas-r3-f16.gguf \
        {GGUF_DIR}/atlas-r3-q4_k_m.gguf Q4_K_M

    gguf_path = f"{GGUF_DIR}/atlas-r3-q4_k_m.gguf"
    print(f"GGUF ready: {gguf_path}")

In [ ]:
# ── Cell 4: Verify model loads ─────────────────────────────────────────────────
from llama_cpp import Llama

llm = Llama(
    model_path=gguf_path,
    n_ctx=4096,
    n_gpu_layers=40,   # T4 has 16 GB — adjust if OOM
    verbose=False,
)

test = llm("What is the CFA Standard II(A)?\n", max_tokens=100)
print(test["choices"][0]["text"])

## MMLU Evaluation — 3 Professional Subsets

In [ ]:
# ── Cell 5: MMLU evaluation ────────────────────────────────────────────────────
from datasets import load_dataset
import json, re
from tqdm import tqdm

MMLU_TASKS = [
    "professional_law",
    "professional_accounting",
    "business_ethics",
]

MMLU_SYSTEM = (
    "You are an expert in professional law, accounting, and business ethics. "
    "Answer multiple choice questions accurately. "
    "Respond with only the letter of the correct answer: A, B, C, or D."
)

def build_mmlu_prompt(row, few_shots):
    prompt = f"<|im_start|>system\n{MMLU_SYSTEM}<|im_end|>\n"
    for ex in few_shots:
        choices = "".join(f"\n{chr(65+i)}. {c}" for i, c in enumerate(ex["choices"]))
        prompt += (
            f"<|im_start|>user\n{ex['question']}{choices}<|im_end|>\n"
            f"<|im_start|>assistant\n{chr(65 + ex['answer'])}<|im_end|>\n"
        )
    choices = "".join(f"\n{chr(65+i)}. {c}" for i, c in enumerate(row["choices"]))
    prompt += f"<|im_start|>user\n{row['question']}{choices}<|im_end|>\n<|im_start|>assistant\n"
    return prompt

def extract_answer(text):
    m = re.search(r"\b([A-D])\b", text.strip().upper())
    return m.group(1) if m else "X"

mmlu_results = {}

for task in MMLU_TASKS:
    ds = load_dataset("cais/mmlu", task, split="test")
    dev = load_dataset("cais/mmlu", task, split="dev")  # 5-shot examples
    few_shots = list(dev)[:5]

    correct = 0
    for row in tqdm(ds, desc=f"MMLU {task}"):
        prompt = build_mmlu_prompt(row, few_shots)
        out = llm(prompt, max_tokens=4, temperature=0.0, stop=["<|"])
        pred = extract_answer(out["choices"][0]["text"])
        correct += int(pred == chr(65 + row["answer"]))

    acc = correct / len(ds)
    mmlu_results[task] = {"accuracy": round(acc, 4), "correct": correct, "total": len(ds)}
    print(f"  {task}: {acc:.1%} ({correct}/{len(ds)})")

print("\n--- MMLU Summary ---")
for t, r in mmlu_results.items():
    print(f"  {t:<30} {r['accuracy']:.1%}")

## LegalBench Evaluation — Compliance Subsets

In [ ]:
# ── Cell 6: LegalBench evaluation ─────────────────────────────────────────────
# Tasks chosen for ATLAS domain fit: contract/compliance/statutory reasoning

LEGALBENCH_TASKS = [
    "contract_nli_explicit_identification",
    "contract_nli_breach_of_contract",
    "textualism_tool_nouns",
    "cuad_anti_assignment",
    "opp115_data_retention",
]

LEGAL_SYSTEM = (
    "You are an expert legal analyst. "
    "Answer legal reasoning questions accurately. "
    "Respond with only: Yes or No."
)

def build_legal_prompt(question, answer_choices=None):
    prompt = f"<|im_start|>system\n{LEGAL_SYSTEM}<|im_end|>\n"
    prompt += f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    return prompt

def extract_yes_no(text):
    t = text.strip().lower()
    if t.startswith("yes"):  return "Yes"
    if t.startswith("no"):   return "No"
    return "No"  # default conservative

legalbench_results = {}

for task in LEGALBENCH_TASKS:
    try:
        ds = load_dataset("nguha/legalbench", task, split="test", trust_remote_code=True)
        # Use up to 100 examples to keep runtime reasonable
        sample = ds.select(range(min(100, len(ds))))

        correct = 0
        for row in tqdm(sample, desc=f"LegalBench {task}"):
            prompt = build_legal_prompt(row["text"])
            out = llm(prompt, max_tokens=8, temperature=0.0, stop=["<|"])
            pred  = extract_yes_no(out["choices"][0]["text"])
            label = str(row.get("answer", row.get("label", ""))).strip()
            correct += int(pred.lower() == label.lower())

        acc = correct / len(sample)
        legalbench_results[task] = {"accuracy": round(acc, 4), "correct": correct, "total": len(sample)}
        print(f"  {task}: {acc:.1%} ({correct}/{len(sample)})")

    except Exception as e:
        print(f"  {task}: SKIP ({e})")

print("\n--- LegalBench Summary ---")
for t, r in legalbench_results.items():
    print(f"  {t:<45} {r['accuracy']:.1%}")

In [ ]:
# ── Cell 7: Save all results + print final report ──────────────────────────────
from datetime import datetime

report = {
    "model":     "Rafaelcedav/atlas-r3-qwen3-14b",
    "quant":     "Q4_K_M",
    "timestamp": datetime.utcnow().isoformat(),
    "mmlu":      mmlu_results,
    "legalbench": legalbench_results,
}

with open("/content/atlas_r3_benchmark_results.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "="*60)
print("  ATLAS R3 — BENCHMARK RESULTS")
print("="*60)
print("\nMMLL (5-shot):")
for t, r in mmlu_results.items():
    bar = "#" * int(r['accuracy'] * 20)
    print(f"  {t:<30} {r['accuracy']:.1%}  [{bar:<20}]")

print("\nLegalBench (0-shot, 100 samples):")
for t, r in legalbench_results.items():
    bar = "#" * int(r['accuracy'] * 20)
    print(f"  {t:<45} {r['accuracy']:.1%}  [{bar:<20}]")

# Overall MMLU average
if mmlu_results:
    mmlu_avg = sum(r['accuracy'] for r in mmlu_results.values()) / len(mmlu_results)
    print(f"\n  MMLU avg (3 subsets): {mmlu_avg:.1%}")

print("\nResults saved to: /content/atlas_r3_benchmark_results.json")
print("Download it and put it in docs/Pruebas/results/ for the submission.")